In [3]:
import pandas as pd
from google.colab import files

uploaded = files.upload()
filename = next(iter(uploaded))

df = pd.read_csv(filename)

print(df.head())
print(df.info())
print(df.isnull().sum())

Saving 00 customer.csv to 00 customer (1).csv
   age  gender   review education purchased
0   30  Female  Average    School        No
1   68  Female     Poor        UG        No
2   70  Female     Good        PG        No
3   72  Female     Good        PG        No
4   16  Female  Average        UG        No
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50 entries, 0 to 49
Data columns (total 5 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   age        50 non-null     int64 
 1   gender     50 non-null     object
 2   review     50 non-null     object
 3   education  50 non-null     object
 4   purchased  50 non-null     object
dtypes: int64(1), object(4)
memory usage: 2.1+ KB
None
age          0
gender       0
review       0
education    0
purchased    0
dtype: int64


In [4]:
from sklearn.preprocessing import OrdinalEncoder

oe = OrdinalEncoder(
    categories=[
        ['Poor', 'Average', 'Good'],
        ['School', 'UG', 'PG']
    ]
)

df[['review', 'education']] = oe.fit_transform(
    df[['review', 'education']]
).astype(int)

print(df.head())

   age  gender  review  education purchased
0   30  Female       1          0        No
1   68  Female       0          1        No
2   70  Female       2          2        No
3   72  Female       2          2        No
4   16  Female       1          1        No


In [5]:
from sklearn.preprocessing import OneHotEncoder

df = pd.read_csv(filename)

ohe = OneHotEncoder(sparse_output=False, drop='first')

gender_encoded = ohe.fit_transform(df[['gender']])

gender_df = pd.DataFrame(
    gender_encoded,
    columns=ohe.get_feature_names_out(['gender'])
)

print(gender_df.head())

   gender_Male
0          0.0
1          0.0
2          0.0
3          0.0
4          0.0


In [6]:
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import (
    OrdinalEncoder,
    OneHotEncoder,
    LabelEncoder
)

df = pd.read_csv(filename)

transformer = [
    ('trans1',
     OrdinalEncoder(categories=[['Poor', 'Average', 'Good']]),
     ['review']),

    ('trans2',
     OrdinalEncoder(categories=[['School', 'UG', 'PG']]),
     ['education']),

    ('trans3',
     OneHotEncoder(sparse_output=False, drop='first'),
     ['gender'])
]

ct = ColumnTransformer(
    transformers=transformer,
    remainder='passthrough'
)

X = df.drop(columns=['purchased'])

X_transformed = ct.fit_transform(X)

X_transformed_df = pd.DataFrame(
    X_transformed,
    columns=ct.get_feature_names_out(),
    index=df.index
)

le = LabelEncoder()
y = le.fit_transform(df['purchased'])

X_transformed_df['purchased'] = y

print(X_transformed_df.head())

   trans1__review  trans2__education  trans3__gender_Male  remainder__age  \
0             1.0                0.0                  0.0            30.0   
1             0.0                1.0                  0.0            68.0   
2             2.0                2.0                  0.0            70.0   
3             2.0                2.0                  0.0            72.0   
4             1.0                1.0                  0.0            16.0   

   purchased  
0          0  
1          0  
2          0  
3          0  
4          0  
